# Java AST → Neo4j Knowledge Graph

`AuctionlandController.java` 를 **JavaParser** 로 파싱해 클래스/필드/메서드/메서드호출/임포트/어노테이션을 추출하고 **Neo4j** 에 지식 그래프로 적재합니다.

- **범위**: `AuctionlandController.java` 1개 파일
- **도구**: JavaParser (`javaparser-core` jar)
- **메서드 호출 해석**: 빠름 (이름 매칭, symbol solver 미사용 → 외부 호출은 `:CallTarget` 노드로 표현)
- **Neo4j**: `neo4j_env.txt` 에서 자격증명 로드

## 셀 구성
1. Python 의존성 설치
2. Portable JDK 다운로드 (시스템 PATH 영향 0, `build/jdk/` 에 압축 해제)
3. JDK 탐지 (portable → PATH → 일반 설치 경로 → IntelliJ jbr → VSCode 번들)
4. 작업 경로 + JavaParser jar 다운로드
5. Java 추출기 소스 적재 / 파일로 출력
6. 컴파일
7. 실행 → JSON
8. 미리보기 (DataFrame)
9. Neo4j 자격증명 로드
10. 연결 + 제약조건
11. 그래프 적재
12. 검증 쿼리
13. Neo4j Browser 시각화 쿼리

In [ ]:
# 1. Python 의존성
%pip install -q neo4j pandas

In [1]:
# 2. Portable JDK 다운로드 (시스템 무영향)
#   - Adoptium API 로 최신 LTS(Temurin 17) Windows x64 zip 을 받아 build/jdk/ 에 풀고
#     그 안의 javac.exe / java.exe 를 사용합니다.
#   - 이미 풀려 있으면 스킵.
import os, json, urllib.request, zipfile, shutil
from pathlib import Path

PROJECT_ROOT = Path(r'c:\AI_Master_Project\auctionland-backend')
JDK_DIR      = PROJECT_ROOT / 'build' / 'jdk'
JDK_DIR.mkdir(parents=True, exist_ok=True)

def _portable_javac():
    hits = list(JDK_DIR.glob('*/bin/javac.exe'))
    return hits[0] if hits else None

PORTABLE_JAVAC = _portable_javac()
UA = {'User-Agent': 'Mozilla/5.0 (auctionland-ast-notebook)'}

def _http_get(url, timeout=60):
    req = urllib.request.Request(url, headers=UA)
    return urllib.request.urlopen(req, timeout=timeout)

def _http_download(url, dest):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=300) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f)

if PORTABLE_JAVAC is None:
    api = ('https://api.adoptium.net/v3/assets/feature_releases/17/ga'
           '?architecture=x64&heap_size=normal&image_type=jdk&os=windows'
           '&project=jdk&vendor=eclipse&page=0&page_size=1')
    print('querying Adoptium ...')
    with _http_get(api) as r:
        data = json.loads(r.read().decode('utf-8'))
    pkg = next(b['package'] for b in data[0]['binaries'] if b['package']['link'].endswith('.zip'))
    zip_path = JDK_DIR / pkg['name']
    print(f"downloading {pkg['link']}")
    print(f"  size: {pkg['size']:,} bytes")
    if not zip_path.exists():
        _http_download(pkg['link'], zip_path)
    print(f'extracting {zip_path.name} ...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(JDK_DIR)
    zip_path.unlink()
    PORTABLE_JAVAC = _portable_javac()
    if PORTABLE_JAVAC is None:
        raise RuntimeError('JDK 압축 해제 후에도 javac.exe 를 찾지 못했습니다.')

PORTABLE_JAVA = PORTABLE_JAVAC.with_name('java.exe')
print(f'portable javac: {PORTABLE_JAVAC}')
print(f'portable java : {PORTABLE_JAVA}')

querying Adoptium ...
downloading https://github.com/adoptium/temurin17-binaries/releases/download/jdk-17.0.18%2B8/OpenJDK17U-jdk_x64_windows_hotspot_17.0.18_8.zip
  size: 190,661,454 bytes
extracting OpenJDK17U-jdk_x64_windows_hotspot_17.0.18_8.zip ...
portable javac: c:\AI_Master_Project\auctionland-backend\build\jdk\jdk-17.0.18+8\bin\javac.exe
portable java : c:\AI_Master_Project\auctionland-backend\build\jdk\jdk-17.0.18+8\bin\java.exe


In [2]:
# 3. JDK 탐지 — portable 우선, 그 다음 PATH / 일반 설치 경로
import shutil, glob

def _find_javac():
    if PORTABLE_JAVAC and PORTABLE_JAVAC.exists():
        return PORTABLE_JAVAC
    p = shutil.which('javac')
    if p:
        return Path(p)
    home = os.environ.get('JAVA_HOME')
    if home:
        cand = Path(home) / 'bin' / 'javac.exe'
        if cand.exists():
            return cand
    patterns = [
        r'C:\Program Files\Java\*\bin\javac.exe',
        r'C:\Program Files\Eclipse Adoptium\*\bin\javac.exe',
        r'C:\Program Files\Microsoft\*\bin\javac.exe',
        r'C:\Program Files\Zulu\*\bin\javac.exe',
        r'C:\Program Files\Amazon Corretto\*\bin\javac.exe',
        r'C:\Program Files\JetBrains\*\jbr\bin\javac.exe',
        os.path.expanduser(r'~\.jdks\*\bin\javac.exe'),
        os.path.expanduser(r'~\.vscode\extensions\redhat.java-*\jre\*\bin\javac.exe'),
    ]
    for pat in patterns:
        hits = glob.glob(pat)
        if hits:
            return Path(sorted(hits)[-1])
    return None

JAVAC = _find_javac()
if JAVAC is None:
    raise RuntimeError('javac 를 찾지 못했습니다 — 셀 2를 다시 실행하세요.')
JAVA = JAVAC.with_name('java.exe')
print(f'javac: {JAVAC}')
print(f'java : {JAVA}')
import subprocess
subprocess.run([str(JAVA), '-version'], check=True)

javac: c:\AI_Master_Project\auctionland-backend\build\jdk\jdk-17.0.18+8\bin\javac.exe
java : c:\AI_Master_Project\auctionland-backend\build\jdk\jdk-17.0.18+8\bin\java.exe


CompletedProcess(args=['c:\\AI_Master_Project\\auctionland-backend\\build\\jdk\\jdk-17.0.18+8\\bin\\java.exe', '-version'], returncode=0)

In [3]:
# 4. 경로 설정 + JavaParser jar 다운로드
TARGET_JAVA  = PROJECT_ROOT / 'src/main/java/com/example/demo/controller/AuctionlandController.java'
WORK_DIR     = PROJECT_ROOT / 'build' / 'ast_extractor'
WORK_DIR.mkdir(parents=True, exist_ok=True)

JAVAPARSER_VERSION = '3.25.10'
JAVAPARSER_JAR = WORK_DIR / f'javaparser-core-{JAVAPARSER_VERSION}.jar'
JAVAPARSER_URL = (
    'https://repo1.maven.org/maven2/com/github/javaparser/javaparser-core/'
    f'{JAVAPARSER_VERSION}/javaparser-core-{JAVAPARSER_VERSION}.jar'
)

if not JAVAPARSER_JAR.exists():
    print(f'Downloading {JAVAPARSER_URL} ...')
    urllib.request.urlretrieve(JAVAPARSER_URL, JAVAPARSER_JAR)

assert TARGET_JAVA.exists(), f'대상 파일이 없습니다: {TARGET_JAVA}'
print(f'jar    : {JAVAPARSER_JAR}  ({JAVAPARSER_JAR.stat().st_size:,} bytes)')
print(f'target : {TARGET_JAVA}')
print(f'work   : {WORK_DIR}')

jar    : c:\AI_Master_Project\auctionland-backend\build\ast_extractor\javaparser-core-3.25.10.jar  (1,414,637 bytes)
target : c:\AI_Master_Project\auctionland-backend\src\main\java\com\example\demo\controller\AuctionlandController.java
work   : c:\AI_Master_Project\auctionland-backend\build\ast_extractor


In [4]:
# (5-pre) JavaParser 추출기 소스 (변수에 적재)
JAVA_SOURCE = r"""import com.github.javaparser.StaticJavaParser;
import com.github.javaparser.ast.CompilationUnit;
import com.github.javaparser.ast.ImportDeclaration;
import com.github.javaparser.ast.NodeList;
import com.github.javaparser.ast.body.*;
import com.github.javaparser.ast.expr.*;

import java.io.File;
import java.nio.file.Files;
import java.nio.file.Paths;
import java.util.ArrayList;
import java.util.List;

public class AstExtractor {

    public static void main(String[] args) throws Exception {
        String filePath = args[0];
        String outPath = args[1];

        CompilationUnit cu = StaticJavaParser.parse(new File(filePath));

        StringBuilder sb = new StringBuilder();
        sb.append("{");
        sb.append("\"file\":").append(s(filePath)).append(",");
        sb.append("\"package\":").append(s(
                cu.getPackageDeclaration().map(p -> p.getNameAsString()).orElse(""))).append(",");

        List<String> imps = new ArrayList<>();
        for (ImportDeclaration imp : cu.getImports()) imps.add(s(imp.getNameAsString()));
        sb.append("\"imports\":[").append(String.join(",", imps)).append("],");

        List<String> classJsons = new ArrayList<>();
        for (ClassOrInterfaceDeclaration cls : cu.findAll(ClassOrInterfaceDeclaration.class)) {
            StringBuilder cb = new StringBuilder();
            cb.append("{");
            cb.append("\"name\":").append(s(cls.getNameAsString())).append(",");
            cb.append("\"isInterface\":").append(cls.isInterface()).append(",");

            List<String> anns = new ArrayList<>();
            for (AnnotationExpr a : cls.getAnnotations()) anns.add(annJson(a));
            cb.append("\"annotations\":[").append(String.join(",", anns)).append("],");

            List<String> fields = new ArrayList<>();
            for (FieldDeclaration f : cls.getFields()) {
                for (VariableDeclarator v : f.getVariables()) {
                    fields.add("{\"name\":" + s(v.getNameAsString())
                            + ",\"type\":" + s(v.getTypeAsString()) + "}");
                }
            }
            cb.append("\"fields\":[").append(String.join(",", fields)).append("],");

            List<String> methods = new ArrayList<>();
            for (MethodDeclaration m : cls.getMethods()) {
                methods.add(methodJson(m.getNameAsString(), m.getTypeAsString(),
                        paramTypes(m.getParameters()), m.getAnnotations(), findCalls(m),
                        m.getBegin().map(p -> p.line).orElse(-1)));
            }
            for (ConstructorDeclaration c : cls.getConstructors()) {
                methods.add(methodJson("<init>", cls.getNameAsString(),
                        paramTypes(c.getParameters()), c.getAnnotations(), findCalls(c),
                        c.getBegin().map(p -> p.line).orElse(-1)));
            }
            cb.append("\"methods\":[").append(String.join(",", methods)).append("]");

            cb.append("}");
            classJsons.add(cb.toString());
        }
        sb.append("\"classes\":[").append(String.join(",", classJsons)).append("]");
        sb.append("}");

        Files.write(Paths.get(outPath), sb.toString().getBytes("UTF-8"));
        System.out.println("OK -> " + outPath);
    }

    static List<String> paramTypes(NodeList<Parameter> ps) {
        List<String> r = new ArrayList<>();
        for (Parameter p : ps) r.add(p.getTypeAsString());
        return r;
    }

    static String methodJson(String name, String returnType, List<String> paramTypes,
                             NodeList<AnnotationExpr> annotations, List<String[]> calls,
                             int line) {
        StringBuilder b = new StringBuilder();
        b.append("{\"name\":").append(s(name)).append(",");
        b.append("\"returnType\":").append(s(returnType)).append(",");
        b.append("\"line\":").append(line).append(",");
        List<String> ps = new ArrayList<>();
        for (String t : paramTypes) ps.add(s(t));
        b.append("\"params\":[").append(String.join(",", ps)).append("],");
        List<String> anns = new ArrayList<>();
        for (AnnotationExpr a : annotations) anns.add(annJson(a));
        b.append("\"annotations\":[").append(String.join(",", anns)).append("],");
        List<String> cs = new ArrayList<>();
        for (String[] c : calls) {
            cs.add("{\"scope\":" + s(c[0]) + ",\"name\":" + s(c[1]) + ",\"argc\":" + c[2] + "}");
        }
        b.append("\"calls\":[").append(String.join(",", cs)).append("]");
        b.append("}");
        return b.toString();
    }

    static List<String[]> findCalls(BodyDeclaration<?> body) {
        List<String[]> calls = new ArrayList<>();
        for (MethodCallExpr mc : body.findAll(MethodCallExpr.class)) {
            String scope = mc.getScope().map(Object::toString).orElse("");
            String name = mc.getNameAsString();
            int argc = mc.getArguments().size();
            calls.add(new String[]{scope, name, String.valueOf(argc)});
        }
        return calls;
    }

    static String annJson(AnnotationExpr a) {
        String name = a.getNameAsString();
        String value = "";
        if (a instanceof SingleMemberAnnotationExpr) {
            value = ((SingleMemberAnnotationExpr) a).getMemberValue().toString();
        } else if (a instanceof NormalAnnotationExpr) {
            value = a.toString();
        }
        return "{\"name\":" + s(name) + ",\"value\":" + s(value) + "}";
    }

    static String s(String v) {
        if (v == null) return "null";
        StringBuilder b = new StringBuilder("\"");
        for (int i = 0; i < v.length(); i++) {
            char c = v.charAt(i);
            switch (c) {
                case '"':  b.append("\\\""); break;
                case '\\': b.append("\\\\"); break;
                case '\n': b.append("\\n"); break;
                case '\r': b.append("\\r"); break;
                case '\t': b.append("\\t"); break;
                default:
                    if (c < 0x20) b.append(String.format("\\u%04x", (int) c));
                    else b.append(c);
            }
        }
        b.append("\"");
        return b.toString();
    }
}
"""
print(f'source size: {len(JAVA_SOURCE):,} chars')

source size: 6,208 chars


In [5]:
# 5. Java 추출기 소스 작성
EXTRACTOR_SRC = WORK_DIR / 'AstExtractor.java'
EXTRACTOR_SRC.write_text(JAVA_SOURCE, encoding='utf-8')
print(f'wrote: {EXTRACTOR_SRC}  ({len(JAVA_SOURCE):,} bytes)')

wrote: c:\AI_Master_Project\auctionland-backend\build\ast_extractor\AstExtractor.java  (6,208 bytes)


In [6]:
# 6. 컴파일
import subprocess
result = subprocess.run(
    [str(JAVAC), '-encoding', 'UTF-8', '-cp', str(JAVAPARSER_JAR), '-d', str(WORK_DIR), str(EXTRACTOR_SRC)],
    capture_output=True, text=True,
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)
result.check_returncode()
print('compiled:', WORK_DIR / 'AstExtractor.class')

STDOUT: 
STDERR: 
compiled: c:\AI_Master_Project\auctionland-backend\build\ast_extractor\AstExtractor.class


In [7]:
# 7. 실행 → JSON
import json
OUT_JSON = WORK_DIR / 'ast.json'
SEP = ';' if os.name == 'nt' else ':'
cp = SEP.join([str(JAVAPARSER_JAR), str(WORK_DIR)])
result = subprocess.run(
    [str(JAVA), '-cp', cp, 'AstExtractor', str(TARGET_JAVA), str(OUT_JSON)],
    capture_output=True, text=True,
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)
result.check_returncode()

with open(OUT_JSON, 'r', encoding='utf-8') as f:
    ast = json.load(f)
print(json.dumps(ast, indent=2, ensure_ascii=False)[:1500])

STDOUT: OK -> c:\AI_Master_Project\auctionland-backend\build\ast_extractor\ast.json

STDERR: 
{
  "file": "c:\\AI_Master_Project\\auctionland-backend\\src\\main\\java\\com\\example\\demo\\controller\\AuctionlandController.java",
  "package": "com.example.demo.controller",
  "imports": [
    "com.example.demo.entity.LocationCode",
    "com.example.demo.repository.LocationCodeRepository",
    "com.example.demo.service.AuctionlandService",
    "org.springframework.http.ResponseEntity",
    "org.springframework.web.bind.annotation.GetMapping",
    "org.springframework.web.bind.annotation.RequestMapping",
    "org.springframework.web.bind.annotation.RequestParam",
    "org.springframework.web.bind.annotation.RestController",
    "java.io.IOException",
    "javax.annotation.PostConstruct",
    "java.util.List",
    "java.util.Optional"
  ],
  "classes": [
    {
      "name": "AuctionlandController",
      "isInterface": false,
      "annotations": [
        {
          "name": "RestControlle

In [8]:
# 8. 미리보기 (DataFrame)
import pandas as pd
from IPython.display import display

print(f"package: {ast['package']}")
print(f"file   : {ast['file']}")

print('\n# Imports')
display(pd.DataFrame({'import': ast['imports']}))

for cls in ast['classes']:
    print(f"\n# Class: {cls['name']}  (interface={cls['isInterface']})")
    print('  Annotations:', [a['name'] for a in cls['annotations']])
    print('  Fields:')
    display(pd.DataFrame(cls['fields']) if cls['fields'] else pd.DataFrame(columns=['name','type']))
    print('  Methods:')
    rows = []
    for m in cls['methods']:
        rows.append({
            'method': m['name'],
            'line': m['line'],
            'returnType': m['returnType'],
            'params': ', '.join(m['params']),
            'annotations': ', '.join(a['name'] for a in m['annotations']),
            'calls': len(m['calls']),
        })
    display(pd.DataFrame(rows))
    print('  CallSites:')
    call_rows = []
    for m in cls['methods']:
        for c in m['calls']:
            call_rows.append({'caller': m['name'], 'scope': c['scope'], 'callee': c['name'], 'argc': c['argc']})
    display(pd.DataFrame(call_rows) if call_rows else pd.DataFrame(columns=['caller','scope','callee','argc']))

package: com.example.demo.controller
file   : c:\AI_Master_Project\auctionland-backend\src\main\java\com\example\demo\controller\AuctionlandController.java

# Imports


,import
0,com.example.demo.entity.LocationCode
1,com.example.demo.repository.LocationCodeReposi...
2,com.example.demo.service.AuctionlandService
3,org.springframework.http.ResponseEntity
4,org.springframework.web.bind.annotation.GetMap...
5,org.springframework.web.bind.annotation.Reques...
6,org.springframework.web.bind.annotation.Reques...
7,org.springframework.web.bind.annotation.RestCo...
8,java.io.IOException
9,javax.annotation.PostConstruct



# Class: AuctionlandController  (interface=False)
  Annotations: ['RestController', 'RequestMapping']
  Fields:


,name,type
0,auctionlandService,AuctionlandService
1,locationCodeRepository,LocationCodeRepository


  Methods:


,method,line,returnType,params,annotations,calls
0,insertLocationCodeInformation,33,void,,PostConstruct,0
1,getSidoLocationNameList,41,ResponseEntity<List<String>>,,GetMapping,1
2,getSiguLocationNameList,47,ResponseEntity<List<String>>,String,GetMapping,1
3,getSidongLocationNameList,53,ResponseEntity<List<String>>,"String, String",GetMapping,1
4,getSiriLocationNameList,59,ResponseEntity<List<String>>,"String, String, String",GetMapping,1
5,getLocationCodeData,66,ResponseEntity<Optional<LocationCode>>,"String, String, String, String",GetMapping,1
6,getAuctionData,81,ResponseEntity<List<String>>,"String, String, String, String",GetMapping,1
7,testCall2,88,String,,GetMapping,1
8,<init>,27,AuctionlandController,"AuctionlandService, LocationCodeRepository",,0


  CallSites:


,caller,scope,callee,argc
0,getSidoLocationNameList,auctionlandService,getSidoLocationNameList,0
1,getSiguLocationNameList,auctionlandService,getSiguLocationNameList,1
2,getSidongLocationNameList,auctionlandService,getSidongLocationNameList,2
3,getSiriLocationNameList,auctionlandService,getSiriLocationNameList,3
4,getLocationCodeData,auctionlandService,getLocationCodeData,4
5,getAuctionData,auctionlandService,getAuctionData,4
6,testCall2,System.out,println,1


In [41]:
# 9. Neo4j 자격증명 로드
NEO4J_ENV = PROJECT_ROOT / 'neo4j_env.txt'
neo4j_cfg = {}
for line in NEO4J_ENV.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    if '=' in line:
        k, v = line.split('=', 1)
        neo4j_cfg[k.strip()] = v.strip()

print({k: ('***' if 'PASSWORD' in k else v) for k, v in neo4j_cfg.items()})

{'NEO4J_URI': 'bolt+ssc://46ada769.databases.neo4j.io', 'NEO4J_USERNAME': '46ada769', 'NEO4J_PASSWORD': '***', 'NEO4J_DATABASE': '46ada769', 'AURA_INSTANCEID': '46ada769', 'AURA_INSTANCENAME': 'AI Master 05_gyoryeong'}


In [36]:
import socket
from urllib.parse import urlparse

uri = neo4j_cfg['NEO4J_URI']
parsed = urlparse(uri)
host = parsed.hostname
port = parsed.port or 7687
print(f'URI   : {uri}')
print(f'host  : {host}')
print(f'port  : {port}')

# DNS
try:
    ip = socket.gethostbyname(host)
    print(f'DNS   : OK -> {ip}')
except Exception as e:
    print(f'DNS   : FAIL — {e}')

# TCP 7687
s = socket.socket()
s.settimeout(5)
try:
    s.connect((host, port))
    print(f'TCP   : OK (포트 {port} 도달)')
except Exception as e:
    print(f'TCP   : FAIL — {e}')
finally:
    s.close()

# 드라이버 핸드셰이크
try:
    driver.verify_connectivity()
    print('verify: OK')
except Exception as e:
    print(f'verify: FAIL — {type(e).__name__}: {e}')


URI   : neo4j+ssc://46ada769.databases.neo4j.io
host  : 46ada769.databases.neo4j.io
port  : 7687
DNS   : OK -> 34.126.171.25
TCP   : OK (포트 7687 도달)
verify: FAIL — DriverError: Driver closed


In [37]:
import traceback
from neo4j import GraphDatabase

try: driver.close()
except: pass

uri      = neo4j_cfg['NEO4J_URI']
password = neo4j_cfg['NEO4J_PASSWORD']
print(f'URI : {uri}')
print(f'PW  : {password[:4]}...{password[-4:]}  (len={len(password)})')

# (1) 사용자명 'neo4j' + bolt+s 직접 연결 (라우팅 우회)
bolt_uri = uri.replace('neo4j+s://', 'bolt+s://')
print(f'\n[1] bolt+s direct, user=neo4j')
try:
    d = GraphDatabase.driver(bolt_uri, auth=('neo4j', password))
    d.verify_connectivity()
    print('   OK')
    d.close()
except Exception as e:
    print(f'   FAIL: {type(e).__name__}: {e}')
    if e.__cause__:
        print(f'   cause: {type(e.__cause__).__name__}: {e.__cause__}')

# (2) 사용자명 '46ada769' (env 원본 값) + bolt+s
print(f'\n[2] bolt+s direct, user=46ada769')
try:
    d = GraphDatabase.driver(bolt_uri, auth=('46ada769', password))
    d.verify_connectivity()
    print('   OK')
    d.close()
except Exception as e:
    print(f'   FAIL: {type(e).__name__}: {e}')
    if e.__cause__:
        print(f'   cause: {type(e.__cause__).__name__}: {e.__cause__}')

# (3) neo4j+ssc (TLS 검증 완화) + 라우팅
ssc_uri = uri.replace('neo4j+s://', 'neo4j+ssc://')
print(f'\n[3] neo4j+ssc routing, user=neo4j')
try:
    d = GraphDatabase.driver(ssc_uri, auth=('neo4j', password))
    d.verify_connectivity()
    print('   OK')
    d.close()
except Exception as e:
    print(f'   FAIL: {type(e).__name__}: {e}')
    if e.__cause__:
        print(f'   cause: {type(e.__cause__).__name__}: {e.__cause__}')


URI : neo4j+ssc://46ada769.databases.neo4j.io
PW  : uT6J...bCo4  (len=43)

[1] bolt+s direct, user=neo4j
   FAIL: AuthError: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}

[2] bolt+s direct, user=46ada769
   OK

[3] neo4j+ssc routing, user=neo4j
   FAIL: AuthError: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}


In [45]:
# 10. Neo4j 연결 + 제약조건
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    neo4j_cfg['NEO4J_URI'],
    auth=(neo4j_cfg['NEO4J_USERNAME'], neo4j_cfg['NEO4J_PASSWORD']),
)
DB_NAME = neo4j_cfg.get('NEO4J_DATABASE', '46ada769')
#DB_NAME = 'neo4j'

CONSTRAINTS = [
    'CREATE CONSTRAINT file_path  IF NOT EXISTS FOR (f:File)        REQUIRE f.path IS UNIQUE',
    'CREATE CONSTRAINT class_fqn  IF NOT EXISTS FOR (c:Class)       REQUIRE c.fqn  IS UNIQUE',
    'CREATE CONSTRAINT method_id  IF NOT EXISTS FOR (m:Method)      REQUIRE m.id   IS UNIQUE',
    'CREATE CONSTRAINT field_id   IF NOT EXISTS FOR (fd:Field)      REQUIRE fd.id  IS UNIQUE',
    'CREATE CONSTRAINT ann_name   IF NOT EXISTS FOR (a:Annotation)  REQUIRE a.name IS UNIQUE',
    'CREATE CONSTRAINT import_fqn IF NOT EXISTS FOR (i:Import)      REQUIRE i.fqn  IS UNIQUE',
    'CREATE CONSTRAINT calltarget_id IF NOT EXISTS FOR (t:CallTarget) REQUIRE t.id IS UNIQUE',
]
with driver.session(database=DB_NAME) as s:
    for q in CONSTRAINTS:
        s.run(q)
    rec = s.run('RETURN "connected to " + $db AS msg', db=DB_NAME).single()
    print(rec['msg'])

connected to 46ada769


In [46]:
# 11. 그래프 적재 (MERGE 기반 idempotent)
def upsert_graph(tx, ast):
    file_path = ast['file']
    pkg = ast['package']
    tx.run('MERGE (f:File {path:$path}) SET f.package=$pkg', path=file_path, pkg=pkg)

    for imp in ast['imports']:
        tx.run('''
            MATCH (f:File {path:$path})
            MERGE (i:Import {fqn:$fqn})
            MERGE (f)-[:IMPORTS]->(i)
        ''', fqn=imp, path=file_path)

    for cls in ast['classes']:
        fqn = (pkg + '.' if pkg else '') + cls['name']
        tx.run('''
            MATCH (f:File {path:$path})
            MERGE (c:Class {fqn:$fqn})
            SET c.name=$name, c.isInterface=$isInterface
            MERGE (f)-[:DECLARES]->(c)
        ''', fqn=fqn, name=cls['name'], isInterface=cls['isInterface'], path=file_path)

        for a in cls['annotations']:
            tx.run('''
                MATCH (c:Class {fqn:$fqn})
                MERGE (an:Annotation {name:$name})
                MERGE (c)-[r:ANNOTATED_WITH]->(an)
                SET r.value=$value
            ''', name=a['name'], value=a['value'], fqn=fqn)

        for fd in cls['fields']:
            fid = f"{fqn}#{fd['name']}"
            tx.run('''
                MATCH (c:Class {fqn:$fqn})
                MERGE (fd:Field {id:$id})
                SET fd.name=$name, fd.type=$type
                MERGE (c)-[:HAS_FIELD]->(fd)
            ''', id=fid, name=fd['name'], type=fd['type'], fqn=fqn)

        for m in cls['methods']:
            sig = f"{m['name']}({','.join(m['params'])})"
            mid = f'{fqn}#{sig}'
            tx.run('''
                MATCH (c:Class {fqn:$fqn})
                MERGE (m:Method {id:$id})
                SET m.name=$name, m.returnType=$returnType, m.signature=$sig, m.line=$line
                MERGE (c)-[:HAS_METHOD]->(m)
            ''', id=mid, name=m['name'], returnType=m['returnType'], sig=sig,
                 line=m['line'], fqn=fqn)

            for a in m['annotations']:
                tx.run('''
                    MATCH (m:Method {id:$mid})
                    MERGE (an:Annotation {name:$name})
                    MERGE (m)-[r:ANNOTATED_WITH]->(an)
                    SET r.value=$value
                ''', name=a['name'], value=a['value'], mid=mid)

            for call in m['calls']:
                target_id = f"{call['scope']}::{call['name']}/{call['argc']}"
                tx.run('''
                    MATCH (m:Method {id:$mid})
                    MERGE (t:CallTarget {id:$tid})
                    SET t.name=$name, t.scope=$scope, t.argc=$argc
                    MERGE (m)-[r:CALLS]->(t)
                    ON CREATE SET r.count=1
                    ON MATCH  SET r.count=coalesce(r.count,0)+1
                ''', tid=target_id, name=call['name'], scope=call['scope'],
                     argc=int(call['argc']), mid=mid)

with driver.session(database=DB_NAME) as s:
    s.execute_write(upsert_graph, ast)
print('graph loaded')

graph loaded


In [47]:
# 12. 검증 쿼리
with driver.session(database=DB_NAME) as s:
    print('# 노드 수 (라벨별)')
    for r in s.run('MATCH (n) UNWIND labels(n) AS l RETURN l, count(*) AS n ORDER BY n DESC'):
        print(f"  {r['l']:15s} {r['n']}")

    print('\n# 관계 수 (타입별)')
    for r in s.run('MATCH ()-[r]->() RETURN type(r) AS t, count(*) AS n ORDER BY n DESC'):
        print(f"  {r['t']:18s} {r['n']}")

    print('\n# AuctionlandController 메서드')
    for r in s.run('''
        MATCH (c:Class {name:'AuctionlandController'})-[:HAS_METHOD]->(m)
        RETURN m.name AS name, m.line AS line, m.returnType AS rt, m.signature AS sig
        ORDER BY line
    '''):
        print(f"  L{r['line']:>3}  {r['name']:30s}  -> {r['rt']:20s}  {r['sig']}")

    print('\n# 메서드 호출 (CALLS)')
    for r in s.run('''
        MATCH (m:Method)-[r:CALLS]->(t:CallTarget)
        RETURN m.name AS caller, t.scope AS scope, t.name AS callee, r.count AS cnt
        ORDER BY caller, callee
    '''):
        print(f"  {r['caller']:30s} -> {r['scope']}.{r['callee']}  (x{r['cnt']})")

# 노드 수 (라벨별)
  Import          12
  Method          9
  CallTarget      7
  Annotation      4
  Field           2
  File            1
  Class           1

# 관계 수 (타입별)
  IMPORTS            12
  ANNOTATED_WITH     10
  HAS_METHOD         9
  CALLS              7
  HAS_FIELD          2
  DECLARES           1

# AuctionlandController 메서드
  L 27  <init>                          -> AuctionlandController  <init>(AuctionlandService,LocationCodeRepository)
  L 33  insertLocationCodeInformation   -> void                  insertLocationCodeInformation()
  L 41  getSidoLocationNameList         -> ResponseEntity<List<String>>  getSidoLocationNameList()
  L 47  getSiguLocationNameList         -> ResponseEntity<List<String>>  getSiguLocationNameList(String)
  L 53  getSidongLocationNameList       -> ResponseEntity<List<String>>  getSidongLocationNameList(String,String)
  L 59  getSiriLocationNameList         -> ResponseEntity<List<String>>  getSiriLocationNameList(String,String,String)
  L 66  getLo

## 13. Neo4j Browser 시각화

Neo4j Browser ([https://browser.neo4j.io](https://browser.neo4j.io)) 또는 Aura Console 에서 아래 쿼리 실행:

```cypher
MATCH (f:File)-[r1:DECLARES]->(c:Class)
OPTIONAL MATCH (c)-[r2:HAS_FIELD]->(fd:Field)
OPTIONAL MATCH (c)-[r3:HAS_METHOD]->(m:Method)
OPTIONAL MATCH (m)-[r4:CALLS]->(t:CallTarget)
OPTIONAL MATCH (c)-[r5:ANNOTATED_WITH]->(ca:Annotation)
OPTIONAL MATCH (m)-[r6:ANNOTATED_WITH]->(ma:Annotation)
OPTIONAL MATCH (f)-[r7:IMPORTS]->(i:Import)
RETURN f, c, fd, m, t, ca, ma, i, r1, r2, r3, r4, r5, r6, r7
```

특정 메서드의 호출 흐름만:

```cypher
MATCH (m:Method)-[:CALLS]->(t:CallTarget)
WHERE m.name STARTS WITH 'get'
RETURN m, t
```

In [48]:
# 종료 — 드라이버 닫기
driver.close()
print('driver closed')

driver closed
